In [ ]:
"""
 [홍석환]
 공백·빈값과 ID 표기를 표준화하고 동일 내용 중복을 제거합니다. 
 진료 및 카탈로그 참조, 서비스 유효기간, 수량·단가·줄 금액과 진료별 상세합을 검증합니다. 
 같은 줄의 값·카탈로그·진료 총액의 근거가 일치할 때만 수량·단가·줄 금액을 복구하고, 변경 이력과 검토 사유를 기록하여 정제본과 DQ 자료를 저장합니다.
 """

# encounter_services: 명세서·ERD 기반 정제
진료별 의료서비스 상세입니다. 명세서 14_진료 서비스 상세, 18_참조 관계, 20_업무규칙·생성근거의 B10·B11·B15 및 ERD의 서비스·진료비·결제 관계를 적용합니다. 원본은 수정하지 않으며 파일 저장·DB 적재는 대기합니다. 모든 코드는 이 노트북에 있습니다. .venv 커널로 위에서부터 실행하세요.

## 1. 데이터 구조: (행, 열) 개수

In [1]:
from pathlib import Path
import csv
import polars as pl
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/input/encounter_services').is_dir())
INPUT = ROOT/'data/input'
COLS = ['encounter_service_id','encounter_id','line_no','service_version_id','quantity','unit_price','line_amount','source_system']
NUMS = ['line_no','quantity','unit_price','line_amount']
files = sorted((INPUT/'encounter_services').glob('*.csv'))
assert files
for p in files:
    with p.open(encoding='utf-8-sig',newline='') as f:
        assert next(csv.reader(f)) == COLS, p.name
raw = pl.concat([pl.scan_csv(p,infer_schema=False,row_index_name='source_record')
    .with_columns(pl.lit(p.name).alias('source_file'),(pl.col('source_record')+1).alias('source_record')) for p in files]).collect()
print('Polars:',pl.__version__,'Files:',len(files),'Original shape:',(raw.height,len(COLS)))
display(raw.group_by('source_file').len().sort('source_file'))

Polars: 2.0.0 Files: 36 Original shape: (3014999, 8)
shape: (36, 2)
┌────────────────────────────────┬───────┐
│ source_file                    ┆ len   │
│ ---                            ┆ ---   │
│ str                            ┆ u32   │
╞════════════════════════════════╪═══════╡
│ encounter_services_2023-01.csv ┆ 85418 │
│ encounter_services_2023-02.csv ┆ 75384 │
│ encounter_services_2023-03.csv ┆ 90464 │
│ encounter_services_2023-04.csv ┆ 75362 │
│ encounter_services_2023-05.csv ┆ 90464 │
│ …                              ┆ …     │
│ encounter_services_2025-08.csv ┆ 85416 │
│ encounter_services_2025-09.csv ┆ 80393 │
│ encounter_services_2025-10.csv ┆ 90434 │
│ encounter_services_2025-11.csv ┆ 75386 │
│ encounter_services_2025-12.csv ┆ 90457 │
└────────────────────────────────┴───────┘


## 2. 데이터 내 컬럼 세부 정보

In [2]:
descriptions = ['서비스 내역 식별자','진료 식별자','진료 내 항목 순번','서비스 버전 식별자','제공 수량','적용 단가','항목 금액','원천 시스템']
column_details = pl.DataFrame({'column':COLS,'description':descriptions,
    'raw_dtype':[str(raw.schema[c]) for c in COLS],
    'target_dtype':['Int64' if c in NUMS else 'String' for c in COLS],
    'null_count':[raw[c].null_count() for c in COLS],
    'unique_count':[raw[c].n_unique() for c in COLS]})
display(column_details)
print('MySQL: ID VARCHAR(32), 금액/순번/수량 INT, source_system VARCHAR(8)')
print('PK: encounter_service_id / 복합 UNIQUE: encounter_id + line_no')
print('B10: 진료일 유효 버전과 동일 단가 / B11: quantity > 0, unit_price > 0, 상세합 = gross_charge')

shape: (8, 6)
┌──────────────────────┬────────────────────┬───────────┬──────────────┬────────────┬──────────────┐
│ column               ┆ description        ┆ raw_dtype ┆ target_dtype ┆ null_count ┆ unique_count │
│ ---                  ┆ ---                ┆ ---       ┆ ---          ┆ ---        ┆ ---          │
│ str                  ┆ str                ┆ str       ┆ str          ┆ i64        ┆ i64          │
╞══════════════════════╪════════════════════╪═══════════╪══════════════╪════════════╪══════════════╡
│ encounter_service_id ┆ 서비스 내역 식별자 ┆ String    ┆ String       ┆ 0          ┆ 3000000      │
│ encounter_id         ┆ 진료 식별자        ┆ String    ┆ String       ┆ 30002      ┆ 1030003      │
│ line_no              ┆ 진료 내 항목 순번  ┆ String    ┆ Int64        ┆ 0          ┆ 3            │
│ service_version_id   ┆ 서비스 버전 식별자 ┆ String    ┆ String       ┆ 0          ┆ 181          │
│ quantity             ┆ 제공 수량          ┆ String    ┆ Int64        ┆ 0          ┆ 3            │
│ unit_p

## 3. 데이터 행(레코드) 샘플

In [3]:
display(raw.head(5))
display(raw.tail(5))
display(raw.sample(n=10,seed=42))

shape: (5, 10)
┌────────────┬───────────┬───────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ source_rec ┆ encounter ┆ encounter ┆ line_no ┆ … ┆ unit_pric ┆ line_amou ┆ source_sy ┆ source_fi │
│ ord        ┆ _service_ ┆ _id       ┆ ---     ┆   ┆ e         ┆ nt        ┆ stem      ┆ le        │
│ ---        ┆ id        ┆ ---       ┆ str     ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ u32        ┆ ---       ┆ str       ┆         ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
│            ┆ str       ┆           ┆         ┆   ┆           ┆           ┆           ┆           │
╞════════════╪═══════════╪═══════════╪═════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1          ┆ ESV_00000 ┆ ENC_00000 ┆ 1       ┆ … ┆ 1000      ┆ 1000      ┆ LEGACY_A  ┆ encounter │
│            ┆ 0001      ┆ 0001      ┆         ┆   ┆           ┆           ┆           ┆ _services │
│            ┆           ┆           ┆         ┆   ┆           ┆           ┆

## 4. 클리닝 전 원본(오염된) 상태
원본 누락·공백·정수 변환 실패·중복·금액 불일치를 진단합니다. 하나의 행이 여러 규칙에 해당할 수 있습니다.

In [4]:
raw_rules = {}
for c in COLS:
    raw_rules['missing_'+c] = pl.col(c).is_null() | (pl.col(c).str.strip_chars()=='')
    raw_rules['whitespace_'+c] = pl.col(c)!=pl.col(c).str.strip_chars()
for c in NUMS:
    s=pl.col(c).str.strip_chars()
    raw_rules['parse_failed_'+c] = s.is_not_null() & (s!='') & s.cast(pl.Int64,strict=False).is_null()
raw_rules['duplicate_content'] = pl.struct(COLS).is_duplicated()
raw_rules['duplicate_id'] = pl.col('encounter_service_id').is_duplicated()
q=pl.col('quantity').cast(pl.Int64,strict=False)
u=pl.col('unit_price').cast(pl.Int64,strict=False)
a=pl.col('line_amount').cast(pl.Int64,strict=False)
raw_rules['amount_mismatch'] = q*u!=a
raw_rules['invalid_quantity'] = q<=0
raw_rules['negative_price_or_amount'] = (u<0)|(a<0)
raw_flags=raw.select([e.fill_null(False).alias(k) for k,e in raw_rules.items()])
raw_report=pl.DataFrame({'rule':list(raw_rules),'rows':[raw_flags[k].sum() for k in raw_rules]}).sort('rows',descending=True)
display(raw_report)
for k in ['missing_encounter_id','amount_mismatch','invalid_quantity','duplicate_content']:
    print(k)
    display(raw.filter(raw_flags[k]).head(3))

shape: (25, 2)
┌──────────────────────────┬───────┐
│ rule                     ┆ rows  │
│ ---                      ┆ ---   │
│ str                      ┆ i64   │
╞══════════════════════════╪═══════╡
│ amount_mismatch          ┆ 30004 │
│ negative_price_or_amount ┆ 30004 │
│ missing_encounter_id     ┆ 30002 │
│ parse_failed_quantity    ┆ 29999 │
│ duplicate_content        ┆ 29998 │
│ …                        ┆ …     │
│ whitespace_source_system ┆ 0     │
│ parse_failed_line_no     ┆ 0     │
│ parse_failed_unit_price  ┆ 0     │
│ parse_failed_line_amount ┆ 0     │
│ invalid_quantity         ┆ 0     │
└──────────────────────────┴───────┘
missing_encounter_id
shape: (3, 10)
┌────────────┬───────────┬───────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ source_rec ┆ encounter ┆ encounter ┆ line_no ┆ … ┆ unit_pric ┆ line_amou ┆ source_sy ┆ source_fi │
│ ord        ┆ _service_ ┆ _id       ┆ ---     ┆   ┆ e         ┆ nt        ┆ stem      ┆ le        │
│ ---        ┆ id

## 5. 클리닝 로직/논리
1. B15에 따라 공백·빈값, ID 하이픈, 참조 날짜 슬래시 표기를 표준화하고 원본 값을 보존합니다.
2. 동일 문자열 내용 중복만 통합합니다. PK 및 진료 ID+순번 충돌은 임의로 고르지 않습니다.
3. 카탈로그의 버전 ID·코드·유효기간·단가·명칭·카테고리를 확인하고, 진료일에 유효한 버전인지 검사합니다. 표준 코드 매핑을 적용합니다.
4. 수량·단가·금액은 MySQL INT 범위로 검사합니다. 수량·단가는 양수, 금액=수량×단가, 단가=해당 버전 단가를 적용합니다.
5. 복구는 같은 줄의 값·카탈로그·진료 총액의 서로 다른 근거가 일치할 때만 수행합니다. 수량은 정상 단가와 금액의 정수 나눗셈 및 진료 상세합 일치로, 단가는 카탈로그와 수량×카탈로그 단가=줄 금액 및 진료 상세합으로 확인합니다.
6. 줄 금액은 정상 수량×버전 단가와 기존 진료 총액−다른 모든 정상 줄의 합이 일치하고 유일한 오염 줄일 때만 복구합니다. 실패한 정수 파싱을 0으로 바꾸지 않습니다.
7. 연결 ID·없는 날짜·복구 후보 충돌은 추측하지 않고 검토 대상으로 남깁니다. 상세는 진료별로 먼저 집계한 뒤 헤더와 대사합니다.
8. 참조 진료는 원본 헤더를 표준화한 것이며 최종 encounters_clean과 동일하지 않습니다. 부모 진료 품질 및 접수·결제 최종 검증은 후속 통합 단계가 필요합니다. 진료당 3줄은 생성 시나리오이므로 누락된 줄을 자동 생성하지 않습니다.

In [5]:
def strip_text(df,columns):
    return df.with_columns([pl.when(pl.col(c).str.strip_chars()=='').then(None).otherwise(pl.col(c).str.strip_chars()).alias(c) for c in columns])
def canonical_ids(df,columns):
    return df.with_columns([pl.col(c).str.replace_all('-','_').alias(c) for c in columns])
def load(table,ids):
    f=sorted((INPUT/table).glob('*.csv'));assert f,table
    r=pl.scan_csv(f,infer_schema=False).collect()
    return canonical_ids(strip_text(r,r.columns),ids).unique(maintain_order=True)
def unique_reference(df,key):
    counts=df.filter(pl.col(key).is_not_null()).group_by(key).len().rename({'len':'reference_count'})
    return counts,df.join(counts.filter(pl.col('reference_count')==1),on=key,validate='m:1').drop('reference_count')
text=canonical_ids(strip_text(raw.with_columns([pl.col(c).alias('raw_'+c) for c in COLS]),COLS),['encounter_service_id','encounter_id','service_version_id'])
duplicate_rows=text.filter(pl.struct(COLS).is_duplicated())
base=text.unique(subset=COLS,keep='first',maintain_order=True).with_columns([pl.col(c).cast(pl.Int64,strict=False) for c in NUMS])
removed=raw.height-base.height
enc=load('encounters',['encounter_id','checkin_id']).with_columns(
    pl.col('started_at').str.replace_all('/','-').str.to_datetime('%Y-%m-%d %H:%M:%S',strict=False).alias('ref_started_at'),
    pl.col('gross_charge').cast(pl.Int64,strict=False).alias('ref_gross_charge'))
enc_counts,enc_unique=unique_reference(enc,'encounter_id')
cat=load('service_catalog',['service_version_id','service_code'])
mappings=load('legacy_code_mappings',['mapping_id'])
map_cat=mappings.filter(pl.col('domain')=='service_category').select('source_system',pl.col('raw_code').alias('service_category'),'standard_code').unique()
map_cat=map_cat.filter(pl.len().over(['source_system','service_category'])==1)
cat=cat.join(map_cat,on=['source_system','service_category'],how='left',validate='m:1').with_columns(pl.coalesce('standard_code','service_category').alias('service_category')).drop('standard_code').unique()
cat=cat.with_columns(pl.col('valid_from').str.replace_all('/','-').str.to_date('%Y-%m-%d',strict=False),pl.col('valid_to').str.replace_all('/','-').str.to_date('%Y-%m-%d',strict=False),pl.col('unit_price').cast(pl.Int64,strict=False))
cat_counts,cat_unique=unique_reference(cat,'service_version_id')
cat_unique=cat_unique.with_columns((pl.col('service_version_id').str.contains(r'^SVV_\d{9}$')&pl.col('service_code').str.contains(r'^SVC_\d{9}$')
    &pl.col('service_name').is_not_null()&pl.col('service_category').is_in(['CONSULT','TEST','PROCEDURE'])
    &(pl.col('unit_price')>0)&(pl.col('unit_price')<=2147483647)&(pl.col('valid_from')<=pl.col('valid_to'))
    &pl.col('source_system').is_in(['LEGACY_A','LEGACY_B'])).fill_null(False).alias('catalog_valid'))
work=base.join(enc_counts.rename({'reference_count':'enc_reference_count'}),on='encounter_id',how='left',validate='m:1').join(
    enc_unique.select('encounter_id','ref_started_at','ref_gross_charge'),on='encounter_id',how='left',validate='m:1').join(
    cat_counts.rename({'reference_count':'cat_reference_count'}),on='service_version_id',how='left',validate='m:1').join(
    cat_unique.select('service_version_id','valid_from','valid_to','catalog_valid',pl.col('unit_price').alias('catalog_price')),on='service_version_id',how='left',validate='m:1')
work=work.with_columns(
    pl.col('ref_started_at').dt.date().is_between(pl.col('valid_from'),pl.col('valid_to'),closed='both').fill_null(False).alias('version_date_valid'),
    (pl.col('encounter_service_id').is_not_null()&(pl.len().over('encounter_service_id')==1)&pl.col('encounter_id').is_not_null()
     &pl.col('line_no').is_not_null()&(pl.col('line_no')>0)&(pl.len().over(['encounter_id','line_no'])==1)).alias('keys_unique'))
# 합계에서 null이 숨지 않도록 전체 행의 누락 수도 집계
sums=work.group_by('encounter_id').agg(pl.col('line_amount').sum().alias('original_line_sum'),pl.col('line_amount').null_count().alias('line_amount_nulls'),pl.col('keys_unique').all().alias('all_keys_unique'))
work=work.join(sums,on='encounter_id',how='left',validate='m:1').with_columns([pl.col(c).alias('before_'+c) for c in ['quantity','unit_price','line_amount']])
header_agrees=(pl.col('enc_reference_count')==1)&pl.col('ref_gross_charge').is_between(0,2147483647)&(pl.col('line_amount_nulls')==0)&(pl.col('original_line_sum')==pl.col('ref_gross_charge'))&pl.col('all_keys_unique')
anchor=pl.col('catalog_valid')&pl.col('version_date_valid')&pl.col('keys_unique')
quantity_candidate=pl.col('line_amount')//pl.col('catalog_price')
quantity_safe=anchor&header_agrees&(pl.col('unit_price')==pl.col('catalog_price'))&(pl.col('line_amount')>0)&((pl.col('line_amount')%pl.col('catalog_price'))==0)&quantity_candidate.is_between(1,2147483647)
work=work.with_columns(pl.when(quantity_safe).then(quantity_candidate).otherwise(pl.col('quantity')).alias('quantity'))
price_safe=anchor&header_agrees&pl.col('quantity').is_between(1,2147483647)&(pl.col('line_amount')==pl.col('quantity')*pl.col('catalog_price'))
work=work.with_columns(pl.when(price_safe).then(pl.col('catalog_price')).otherwise(pl.col('unit_price')).alias('unit_price'))
work=work.with_columns((anchor&pl.col('quantity').is_between(1,2147483647)&(pl.col('unit_price')==pl.col('catalog_price'))
    &(pl.col('line_amount')==pl.col('quantity')*pl.col('unit_price'))&pl.col('line_amount').is_between(1,2147483647)).fill_null(False).alias('line_evidence_valid'))
peer=work.group_by('encounter_id').agg(pl.when(pl.col('line_evidence_valid')).then(pl.col('line_amount')).otherwise(0).sum().alias('valid_line_sum'),(~pl.col('line_evidence_valid')).sum().alias('invalid_line_count'))
work=work.join(peer,on='encounter_id',how='left',validate='m:1')
amount_candidate=pl.col('quantity')*pl.col('catalog_price')
amount_safe=anchor&pl.col('all_keys_unique')&(pl.col('enc_reference_count')==1)&(pl.col('invalid_line_count')==1)&(~pl.col('line_evidence_valid'))
amount_safe=amount_safe&pl.col('quantity').is_between(1,2147483647)&(pl.col('unit_price')==pl.col('catalog_price'))&amount_candidate.is_between(1,2147483647)&((pl.col('ref_gross_charge')-pl.col('valid_line_sum'))==amount_candidate)
work=work.with_columns(pl.when(amount_safe).then(amount_candidate).otherwise(pl.col('line_amount')).alias('line_amount'))
audit=[]
for c in ['quantity','unit_price','line_amount']:
    audit.append(work.filter(~pl.col(c).eq_missing(pl.col('before_'+c))).select('encounter_service_id','encounter_id','source_file','source_record',pl.lit(c).alias('column'),pl.col('raw_'+c).alias('original_value'),pl.col(c).cast(pl.String).alias('clean_value'),pl.lit('catalog + line equation + encounter gross cross-check').alias('evidence')))
for c in ['encounter_service_id','encounter_id','service_version_id','source_system']:
    audit.append(text.filter(~pl.col(c).eq_missing(pl.col('raw_'+c))).select('encounter_service_id','encounter_id','source_file','source_record',pl.lit(c).alias('column'),pl.col('raw_'+c).alias('original_value'),pl.col(c).alias('clean_value'),pl.lit('B15 notation normalization').alias('evidence')))
repair_log=pl.concat(audit,how='vertical_relaxed')
rules={}
for c in COLS:rules['missing_or_unparseable_'+c]=pl.col(c).is_null()
for c,prefix in [('encounter_service_id','ESV'),('encounter_id','ENC'),('service_version_id','SVV')]:rules['invalid_'+c]=~pl.col(c).str.contains('^'+prefix+r'_\d{9}$')
rules.update({
    'duplicate_service_id':pl.col('encounter_service_id').is_not_null()&(pl.len().over('encounter_service_id')>1),
    'duplicate_encounter_line':pl.col('encounter_id').is_not_null()&pl.col('line_no').is_not_null()&(pl.len().over(['encounter_id','line_no'])>1),
    'invalid_source_system':~pl.col('source_system').is_in(['LEGACY_A','LEGACY_B']),
    'invalid_integer_range':pl.any_horizontal([~pl.col(c).is_between(1,2147483647) for c in NUMS]),
    'amount_mismatch':pl.col('line_amount')!=pl.col('quantity')*pl.col('unit_price'),
    'catalog_price_mismatch':pl.col('unit_price')!=pl.col('catalog_price'),
    'unknown_encounter':pl.col('encounter_id').is_not_null()&pl.col('enc_reference_count').is_null(),
    'ambiguous_encounter':pl.col('enc_reference_count')>1,
    'unknown_service_version':pl.col('service_version_id').is_not_null()&pl.col('cat_reference_count').is_null(),
    'ambiguous_service_version':pl.col('cat_reference_count')>1,
    'catalog_evidence_invalid':(pl.col('cat_reference_count')==1)&(~pl.col('catalog_valid')),
    'invalid_encounter_date':(pl.col('enc_reference_count')==1)&pl.col('ref_started_at').is_null(),
    'invalid_catalog_dates':(pl.col('cat_reference_count')==1)&(pl.col('valid_from').is_null()|pl.col('valid_to').is_null()|(pl.col('valid_from')>pl.col('valid_to'))),
    'outside_service_validity':(pl.col('ref_started_at').dt.date()<pl.col('valid_from'))|(pl.col('ref_started_at').dt.date()>pl.col('valid_to')),
})
work=work.with_columns([e.fill_null(False).alias(k) for k,e in rules.items()]).with_columns(pl.concat_list([pl.when(pl.col(k)).then(pl.lit(k)).otherwise(None) for k in rules]).list.drop_nulls().alias('row_issue_reasons'))
# 전체 상세를 집계하고 검증 미완료 그룹과 금액 불일치를 구분
reconciliation=work.filter(pl.col('encounter_id').is_not_null()).group_by('encounter_id').agg(pl.col('line_amount').sum().alias('service_total'),pl.col('line_amount').null_count().alias('amount_nulls'),(pl.col('row_issue_reasons').list.len()>0).sum().alias('invalid_rows'))
reconciliation=enc_unique.select('encounter_id','ref_gross_charge').join(reconciliation,on='encounter_id',how='left',validate='1:1').with_columns(
    pl.when(pl.col('service_total').is_null()).then(pl.lit('missing_details'))
    .when(pl.col('ref_gross_charge').is_null()|(pl.col('amount_nulls')>0)|(pl.col('invalid_rows')>0)).then(pl.lit('unverified'))
    .when(pl.col('service_total')!=pl.col('ref_gross_charge')).then(pl.lit('mismatch'))
    .otherwise(pl.lit('match')).alias('reconciliation_status'))
checked=work.join(reconciliation.select('encounter_id','reconciliation_status'),on='encounter_id',how='left',validate='m:1')
checked=checked.with_columns(pl.concat_list(pl.col('row_issue_reasons'),
    pl.when(pl.col('reconciliation_status')=='mismatch').then(pl.lit('encounter_total_mismatch')).otherwise(None),
    pl.when(pl.col('reconciliation_status')=='unverified').then(pl.lit('encounter_group_unverified')).otherwise(None)).list.drop_nulls().alias('issue_reasons'))
quality_report=pl.DataFrame({'rule':list(rules),'rows':[checked[k].sum() for k in rules]}).sort('rows',descending=True)
reconciliation_report=reconciliation.group_by('reconciliation_status').len()

## 6. 클리닝 결과
정상 줄의 국소 검증과 진료 전체 상세합 검증을 구분합니다. 같은 진료의 다른 줄이 오염되면 전체 대사가 미완료이므로 그룹 검토 사유를 추가합니다. 부모 진료의 접수·결제 검증은 최종 통합 단계에서 재평가해야 합니다.

In [6]:
encounter_services_clean=checked.filter(pl.col('issue_reasons').list.len()==0).select(COLS+['source_file','source_record'])
encounter_services_review=checked.filter(pl.col('issue_reasons').list.len()>0)
assert raw.height==removed+encounter_services_clean.height+encounter_services_review.height
assert encounter_services_clean['encounter_service_id'].n_unique()==encounter_services_clean.height
assert encounter_services_clean.select('encounter_id','line_no').unique().height==encounter_services_clean.height
assert encounter_services_clean.select(pl.col(COLS).null_count()).row(0)==(0,)*len(COLS)
summary=pl.DataFrame({'stage':['raw','duplicate_removed','clean','review'],'rows':[raw.height,removed,encounter_services_clean.height,encounter_services_review.height]})
repair_summary=repair_log.group_by('column','evidence').len().sort('column')
display(summary)
display(quality_report)
display(repair_summary)
display(repair_log.head(10))
display(reconciliation_report)
display(encounter_services_clean.head(5))
display(encounter_services_review.select(COLS+['issue_reasons']).head(10))

shape: (4, 2)
┌───────────────────┬─────────┐
│ stage             ┆ rows    │
│ ---               ┆ ---     │
│ str               ┆ i64     │
╞═══════════════════╪═════════╡
│ raw               ┆ 3014999 │
│ duplicate_removed ┆ 14999   │
│ clean             ┆ 2417760 │
│ review            ┆ 582240  │
└───────────────────┴─────────┘
shape: (25, 2)
┌─────────────────────────────────┬────────┐
│ rule                            ┆ rows   │
│ ---                             ┆ ---    │
│ str                             ┆ i64    │
╞═════════════════════════════════╪════════╡
│ catalog_evidence_invalid        ┆ 132122 │
│ catalog_price_mismatch          ┆ 66054  │
│ missing_or_unparseable_encount… ┆ 30002  │
│ unknown_service_version         ┆ 29999  │
│ invalid_encounter_date          ┆ 29988  │
│ …                               ┆ …      │
│ unknown_encounter               ┆ 0      │
│ ambiguous_encounter             ┆ 0      │
│ ambiguous_service_version       ┆ 0      │
│ invalid_catalog_dat

## 7. 정제 결과 파일 저장
최종 정제 데이터는 data/output/encounter_services.csv에 업무 컬럼만 저장합니다. Polars는 pandas의 인덱스와 같은 별도 번호 컬럼을 생성하지 않습니다. 추적용 source_record도 최종 파일에서 제외합니다.

UTF-8 BOM(utf-8-sig), 날짜 YYYY-MM-DD, 일시 YYYY-MM-DD HH:MM:SS를 사용하고 실제 결측값은 빈칸으로 저장합니다. 검토 대상·변경 이력·검증 보고서는 data/output/reports/encounter_services/에 같은 기준으로 저장합니다. 원본 값을 기록한 검토 자료에는 오염된 문자열이 그대로 남을 수 있습니다.

각 노트북은 기존 정제 규칙으로 독립 실행하며, 다른 노트북의 복구 결과가 자동 연결되는 것은 아닙니다. MySQL 적재는 수행하지 않습니다.

최종 정제본에서 제외된 검토 대상과 실제 중복 제거 행은 data/output/encounter_services_dq.csv에 합쳐 저장합니다. dq_type은 REVIEW 또는 DUPLICATE_EXCLUDED, dq_reason은 검토 사유입니다. 업무 컬럼에는 수정 전 원본 값을 넣고 파일명과 레코드 순번을 남깁니다. 중복 그룹의 대표 행은 중복 제외 목록에 넣지 않습니다. 제외 행이 없더라도 헤더가 있는 빈 파일을 생성합니다.

In [7]:
from pathlib import Path
import json
FINAL_DIR = ROOT / 'data/output'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = FINAL_DIR / 'reports' / 'encounter_services'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def export_csv(frame, filename):
    # CSV에 중첩 자료형을 직접 저장할 수 없어 사유 목록을 구분 문자열로 보존
    out = frame.with_columns([pl.col(c).list.join('|').alias(c)
        for c, dtype in frame.schema.items() if isinstance(dtype, pl.List)])
    destination = FINAL_DIR / filename if filename in ['encounter_services.csv', 'encounter_services_dq.csv'] else OUTPUT_DIR / filename
    out.write_csv(destination, include_bom=True, null_value='',
        datetime_format='%Y-%m-%d %H:%M:%S', date_format='%Y-%m-%d')
    return {'file': filename, 'rows': frame.height, 'columns': frame.width}
exported = []
business_columns = COLS
exported.append(export_csv(encounter_services_clean.select(business_columns), 'encounter_services.csv'))
# DQ에는 검토 행과 실제 제거된 중복 행을 원본 값으로 기록
provenance = ['source_file', 'source_record']
retained_locations = base.select(provenance)
excluded_locations = text.select(provenance).join(retained_locations, on=provenance, how='anti')
review_reasons = encounter_services_review.select(provenance + ['issue_reasons']).with_columns(
    pl.col('issue_reasons').list.join('|').alias('dq_reason')).drop('issue_reasons')
dq_review = raw.join(review_reasons, on=provenance, how='inner', validate='1:1').with_columns(
    pl.lit('REVIEW').alias('dq_type'))
dq_duplicates = raw.join(excluded_locations, on=provenance, how='inner', validate='1:1').with_columns(
    pl.lit('DUPLICATE_EXCLUDED').alias('dq_type'),
    pl.lit('identical_normalized_content').alias('dq_reason'))
dq_columns = business_columns + provenance + ['dq_type', 'dq_reason']
encounter_services_dq = pl.concat([dq_review.select(dq_columns), dq_duplicates.select(dq_columns)])
expected_excluded = summary.filter(pl.col('stage') == 'duplicate_removed')['rows'][0]
assert dq_duplicates.height == expected_excluded
assert dq_review.height == encounter_services_review.height
assert raw.height == encounter_services_clean.height + encounter_services_dq.height
assert encounter_services_dq.select(provenance).unique().height == encounter_services_dq.height
assert encounter_services_dq.select(provenance).join(encounter_services_clean.select(provenance), on=provenance, how='inner').is_empty()
exported.append(export_csv(encounter_services_dq, 'encounter_services_dq.csv'))
loaded_dq = pl.read_csv(FINAL_DIR / 'encounter_services_dq.csv', infer_schema=False)
assert loaded_dq.height == encounter_services_dq.height and loaded_dq.columns == dq_columns

exported.append(export_csv(encounter_services_review, 'review.csv'))
for variable in ['summary', 'quality_report', 'raw_report', 'repair_log', 'repair_summary',
                 'reconciliation', 'reconciliation_report', 'reference_report',
                 'duplicate_rows', 'seen_without_encounter']:
    value = globals().get(variable)
    if isinstance(value, pl.DataFrame):
        exported.append(export_csv(value, variable + '.csv'))
# 전송 결과의 행 수 및 스키마 확인 (자료형은 CSV 밖 manifest에 기록)
loaded = pl.read_csv(FINAL_DIR / 'encounter_services.csv', infer_schema=False)
assert loaded.height == encounter_services_clean.height and loaded.columns == business_columns
manifest = {'table': 'encounter_services', 'polars_version': pl.__version__,
    'clean_schema': {c: str(dtype) for c, dtype in encounter_services_clean.select(business_columns).schema.items()},
    'null_token': '', 'list_separator': '|', 'encoding': 'utf-8-sig',
    'execution_mode': 'independent notebooks using original reference data', 'files': exported}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Final CSV:', FINAL_DIR / 'encounter_services.csv')
print('Reports:', OUTPUT_DIR)
display(pl.DataFrame(exported))


Final CSV: C:\workspace\medallion_venv\data\output\encounter_services.csv
Reports: C:\workspace\medallion_venv\data\output\reports\encounter_services
shape: (11, 3)
┌───────────────────────────┬─────────┬─────────┐
│ file                      ┆ rows    ┆ columns │
│ ---                       ┆ ---     ┆ ---     │
│ str                       ┆ i64     ┆ i64     │
╞═══════════════════════════╪═════════╪═════════╡
│ encounter_services.csv    ┆ 2417760 ┆ 8       │
│ encounter_services_dq.csv ┆ 597239  ┆ 12      │
│ review.csv                ┆ 582240  ┆ 65      │
│ summary.csv               ┆ 4       ┆ 2       │
│ quality_report.csv        ┆ 25      ┆ 2       │
│ …                         ┆ …       ┆ …       │
│ repair_log.csv            ┆ 84652   ┆ 8       │
│ repair_summary.csv        ┆ 3       ┆ 3       │
│ reconciliation.csv        ┆ 1000000 ┆ 6       │
│ reconciliation_report.csv ┆ 3       ┆ 2       │
│ duplicate_rows.csv        ┆ 29998   ┆ 18      │
└───────────────────────────┴──────